In [ ]:
import pandas as pd 

df = pd.read_csv('/kaggle/input/forest-cover-type-prediction/train.csv')
df

In [ ]:
print(df.shape)
print(df.columns)
print(df.dtypes)


In [ ]:
df.nunique()

In [ ]:
print(df.isna().sum())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


import numpy as np

# Get absolute correlations and find strongest ones
corr = df.corr().abs()
upper = corr.where(np.triu(np.ones_like(corr, dtype=bool)))
strong_pairs = (upper.stack()
                .sort_values(ascending=False)
                .head(20))  # Top 20 correlations

# Plot only the strongest correlations
plt.figure(figsize=(10, 8))
sns.heatmap(df[strong_pairs.index.get_level_values(0).unique()].corr(),
            annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Top Correlated Features")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split


# Separate features and target
X = df.drop('Cover_Type', axis=1)
y = df['Cover_Type']

# Handle categorical variables (if any)
X = pd.get_dummies(X, drop_first=True)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, mean_squared_error

# For classification
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

rf.fit(X_train, y_train)

# Feature Importance

In [ ]:
# Get feature importances
importances = rf.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# Plot top N features
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', 
           data=feature_importance_df.head(20))
plt.title('Top 20 Important Features')
plt.show()

# Feature Selection

In [ ]:
# Select top K features based on importance
K = 30  # Adjust based on your needs
selected_features = feature_importance_df['Feature'].head(K).values
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

In [ ]:
selected_features

# Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(estimator=rf,
                         param_grid=param_grid,
                         cv=5,
                         n_jobs=-1,
                         verbose=2)

grid_search.fit(X_train_selected, y_train)
best_rf = grid_search.best_estimator_

In [ ]:
best_rf

# Model Evaluation

In [ ]:
# For classification
y_pred = best_rf.predict(X_test_selected)
print(classification_report(y_test, y_pred))

# Feature importance plot for final model
plt.figure(figsize=(10, 6))
pd.Series(best_rf.feature_importances_, index=selected_features).sort_values().plot(kind='barh')
plt.title('Final Model Feature Importance')
plt.show()